# Deadeye rim detector: training

Trains a small model that finds the **basketball rim, the backboard and the ball** in a photo, so the shot tracker can
find the hoop by itself instead of the player tapping its two sides.

**Before you start (once):**
1. In Google Drive, make a folder called **`deadeye-rim`** (in My Drive).
2. Put these zips in it, from `Documents/deadeye-bench/datasets` on the laptop:
   `ballllls.coco.zip`, `X-Basket.coco.zip`, `basketball-computer-vision.coco.zip`,
   `Basketball Shot Tracking.coco.zip` and `deadeye-eval-frames.zip`.

**To run:**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Allow Google Drive access when asked.
3. It takes about 1 to 2 hours. If Colab disconnects, just **Run all** again: training carries on from where it stopped.
4. When it finishes, download **`deadeye-rim-detector-result.zip`** from the `deadeye-rim` folder in Drive into
   `Documents/deadeye-bench` on the laptop, and tell Claude.

Model: RF-DETR Nano (Apache 2.0). Datasets: CC BY 4.0 and public domain, from Roboflow Universe (credits in
`ml/rim-detector/ATTRIBUTION.md` in the Deadeye repo).

In [ ]:
# 1. Setup: GPU check, packages, Google Drive
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU:", gpu.stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rfdetr[train,loggers,onnx]==1.11.2", "onnxruntime", "supervision"], check=True)

from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/deadeye-rim"
OUT = os.path.join(DRIVE, "training")          # checkpoints live on Drive, so a disconnect loses nothing
os.makedirs(OUT, exist_ok=True)
print("Drive folder:", DRIVE)
print("Files there:", sorted(os.listdir(DRIVE)))

In [ ]:
# 2. Settings. The defaults are what Claude planned; change only if asked.
MODEL_SIZE = "nano"          # "nano" (smallest, for phones) or "small"
EPOCHS = 40
BATCH_SIZE = 8
GRAD_ACCUM = 2
LEARNING_RATE = 1e-4

DATASETS = {
    # zip name in Drive -> short name used in file names
    "ballllls.coco.zip": "ballllls",
    "X-Basket.coco.zip": "xbasket",
    "basketball-computer-vision.coco.zip": "bcv",
    "Basketball Shot Tracking.coco.zip": "bst",
}
EVAL_ZIP = "deadeye-eval-frames.zip"

# Every source label, mapped onto Deadeye's three classes. Anything not listed is dropped:
# players, referees, scores, and "basket" (ballllls labels the rim and the basket separately).
CLASS_MAP = {
    "ball": "ball", "basketball": "ball", "Basketball": "ball",
    "rim": "rim", "Basketball-Rim": "rim",
    "board": "backboard", "Basketball-backboard": "backboard",
}
CLASSES = ["ball", "rim", "backboard"]          # category ids 1, 2, 3; id 0 is the supercategory, as Roboflow does

missing = [z for z in list(DATASETS) + [EVAL_ZIP] if not os.path.exists(os.path.join(DRIVE, z))]
if missing:
    raise SystemExit(f"Missing from Drive/deadeye-rim: {missing}")
print("All zips found.")

In [ ]:
# 3. Unzip and merge the datasets into one, with the three classes
import json, shutil, zipfile
from collections import Counter

SRC = "/content/src"
MERGED = "/content/merged"
if os.path.exists(MERGED):
    shutil.rmtree(MERGED)
for zname, short in DATASETS.items():
    dest = os.path.join(SRC, short)
    if not os.path.exists(dest):
        print("Unzipping", zname)
        with zipfile.ZipFile(os.path.join(DRIVE, zname)) as z:
            z.extractall(dest)

categories = [{"id": 0, "name": "basketball-objects", "supercategory": "none"}] + [
    {"id": i + 1, "name": name, "supercategory": "basketball-objects"} for i, name in enumerate(CLASSES)
]
cat_id = {name: i + 1 for i, name in enumerate(CLASSES)}
report = {}
for split in ["train", "valid", "test"]:
    images, annotations = [], []
    counts = Counter()
    os.makedirs(os.path.join(MERGED, split), exist_ok=True)
    for short in DATASETS.values():
        ann_path = os.path.join(SRC, short, split, "_annotations.coco.json")
        if not os.path.exists(ann_path):
            continue
        with open(ann_path) as f:
            coco = json.load(f)
        src_names = {c["id"]: c["name"] for c in coco["categories"]}
        new_image_id = {}
        for im in coco["images"]:
            src_file = os.path.join(SRC, short, split, im["file_name"])
            if not os.path.exists(src_file):
                continue
            new_id = len(images) + 1
            new_name = f"{short}_{im['file_name']}"
            shutil.copy(src_file, os.path.join(MERGED, split, new_name))
            new_image_id[im["id"]] = new_id
            images.append({"id": new_id, "file_name": new_name, "width": im["width"], "height": im["height"]})
        for a in coco["annotations"]:
            target = CLASS_MAP.get(src_names.get(a["category_id"], ""))
            if target is None or a["image_id"] not in new_image_id:
                continue
            x, y, w, h = a["bbox"]
            if w < 1 or h < 1:
                continue
            annotations.append({
                "id": len(annotations) + 1, "image_id": new_image_id[a["image_id"]], "category_id": cat_id[target],
                "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0, "segmentation": [],
            })
            counts[target] += 1
    with open(os.path.join(MERGED, split, "_annotations.coco.json"), "w") as f:
        json.dump({"info": {"description": "Deadeye merged ball/rim/backboard"}, "images": images,
                   "annotations": annotations, "categories": categories}, f)
    report[split] = {"images": len(images), **counts}
    print(f"{split:6s} {len(images):5d} images  " + "  ".join(f"{c}: {counts[c]}" for c in CLASSES))
if report["valid"]["images"] == 0 or report["train"]["images"] == 0:
    raise SystemExit("The merged dataset has no train or valid images; check the zips.")

In [ ]:
# 4. Train (resumes automatically after a disconnect)
from rfdetr import RFDETRNano, RFDETRSmall
Model = RFDETRNano if MODEL_SIZE == "nano" else RFDETRSmall

best = os.path.join(OUT, "checkpoint_best_total.pth")
last = os.path.join(OUT, "checkpoint.pth")
done_marker = os.path.join(OUT, "TRAINING_DONE")
if os.path.exists(done_marker) and os.path.exists(best):
    print("Training already finished earlier; skipping to evaluation.")
else:
    model = Model()
    kwargs = dict(dataset_dir=MERGED, epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM,
                  lr=LEARNING_RATE, output_dir=OUT, early_stopping=True, early_stopping_patience=8)
    if os.path.exists(last):
        print("Resuming from", last)
        kwargs["resume"] = last
    model.train(**kwargs)
    open(done_marker, "w").write("done")
print("Best checkpoint:", best, os.path.exists(best))

In [ ]:
# 5. How well it does on the held-back test images (none of these were trained on)
import numpy as np
from PIL import Image

model = Model(pretrain_weights=best)
RESOLUTION = 384 if MODEL_SIZE == "nano" else 512

# The class names the model gives its finds. Taken from the finds themselves where the package
# provides them, so nothing here depends on how it numbers the classes.
raw_names = getattr(model, "class_names", None)
print("Model's classes:", raw_names)

def class_name(cid):
    if isinstance(raw_names, dict) and cid in raw_names:
        return raw_names[cid]
    if isinstance(raw_names, (list, tuple)) and 0 <= cid < len(raw_names):
        return raw_names[cid]
    return CLASSES[cid - 1] if 1 <= cid <= len(CLASSES) else str(cid)

def det_names(det):
    names = det.data.get("class_name") if hasattr(det, "data") and det.data else None
    if names is not None and len(names) == len(det):
        return [str(n) for n in names]
    return [class_name(int(c)) for c in det.class_id.tolist()]

with open(os.path.join(MERGED, "test", "_annotations.coco.json")) as f:
    test = json.load(f)
by_image = {}
for a in test["annotations"]:
    by_image.setdefault(a["image_id"], []).append(a)

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

THRESHOLD = 0.5
stats = {c: {"tp": 0, "fp": 0, "fn": 0} for c in CLASSES}
for im in test["images"]:
    det = model.predict(Image.open(os.path.join(MERGED, "test", im["file_name"])).convert("RGB"), threshold=THRESHOLD)
    truths = [(CLASSES[a["category_id"] - 1], [a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]])
              for a in by_image.get(im["id"], [])]
    used = set()
    for box, name in zip(det.xyxy.tolist(), det_names(det)):
        if name not in stats:
            continue
        match = next((i for i, (tc, tb) in enumerate(truths) if i not in used and tc == name and iou(box, tb) >= 0.5), None)
        if match is None:
            stats[name]["fp"] += 1
        else:
            used.add(match); stats[name]["tp"] += 1
    for i, (tc, _) in enumerate(truths):
        if i not in used:
            stats[tc]["fn"] += 1

test_report = {}
print(f"Test images: {len(test['images'])}  (a find counts when its box overlaps the label by half or more)")
for c in CLASSES:
    s = stats[c]
    precision = s["tp"] / max(1, s["tp"] + s["fp"]); recall = s["tp"] / max(1, s["tp"] + s["fn"])
    test_report[c] = {**s, "precision": round(precision, 3), "recall": round(recall, 3)}
    print(f"  {c:9s} found {recall:6.1%} of them; {precision:6.1%} of its finds were right")

In [ ]:
# 6. The real question: does it find the rim on Deadeye's own footage?
# Each frame has the rim's two edges tapped by hand. We compare the detected rim box's left and right sides
# with those taps, in rim widths: under 0.15 is about as good as a careful tap.
EVAL = "/content/eval"
if not os.path.exists(EVAL):
    with zipfile.ZipFile(os.path.join(DRIVE, EVAL_ZIP)) as z:
        z.extractall(EVAL)
with open(os.path.join(EVAL, "frames.json")) as f:
    frames = json.load(f)["frames"]

from PIL import ImageDraw
PREVIEWS = os.path.join(OUT, "previews")
os.makedirs(PREVIEWS, exist_ok=True)
rows = []
for fr in frames:
    img = Image.open(os.path.join(EVAL, fr["file"])).convert("RGB")
    W, H = img.size
    x1, y1, x2, y2 = fr["rim"]
    tl, tr = x1 * W, x2 * W
    width = tr - tl
    det = model.predict(img, threshold=0.2)
    rims = [(conf, box) for box, name, conf in zip(det.xyxy.tolist(), det_names(det), det.confidence.tolist())
            if name == "rim"]
    rims.sort(key=lambda r: -r[0])
    row = {"file": fr["file"], "found": bool(rims)}
    draw = ImageDraw.Draw(img)
    draw.line([(tl, y1 * H), (tr, y2 * H)], fill=(0, 255, 0), width=3)
    if rims:
        conf, box = rims[0]
        edge_err = (abs(box[0] - tl) + abs(box[2] - tr)) / 2 / width
        centre_y = (y1 + y2) / 2 * H
        on_target = box[0] - width < (tl + tr) / 2 < box[2] + width and box[1] - width < centre_y < box[3] + width
        row.update(confidence=round(conf, 3), edge_error_rims=round(edge_err, 3), right_hoop=bool(on_target), rims_found=len(rims))
        draw.rectangle(box, outline=(255, 120, 0), width=3)
    rows.append(row)
    img.save(os.path.join(PREVIEWS, fr["file"]))

found = [r for r in rows if r["found"]]
right = [r for r in found if r["right_hoop"]]
errs = sorted(r["edge_error_rims"] for r in right)
print(f"Rim found in {len(found)}/{len(rows)} Deadeye frames; the right hoop in {len(right)}/{len(rows)}.")
if errs:
    print(f"Edge error on the right hoop, in rim widths: median {errs[len(errs)//2]:.3f}, worst {errs[-1]:.3f}")
for r in rows:
    print(" ", r)
print("Previews (green = hand taps, orange = model) saved to Drive:", PREVIEWS)

In [ ]:
# 7. Export for the browser (ONNX), and a smaller int8 copy
import glob
ONNX_DIR = os.path.join(OUT, "onnx")
os.makedirs(ONNX_DIR, exist_ok=True)
model.export(output_dir=ONNX_DIR)
onnx_files = [p for p in glob.glob(os.path.join(ONNX_DIR, "*.onnx")) if "backbone" not in p and "int8" not in p]
if not onnx_files:
    raise SystemExit(f"Export wrote no .onnx file into {ONNX_DIR}: {os.listdir(ONNX_DIR)}")
ONNX = onnx_files[0]
print("Exported:", ONNX, f"{os.path.getsize(ONNX) / 1e6:.1f} MB")

from onnxruntime.quantization import quantize_dynamic, QuantType
ONNX_INT8 = ONNX.replace(".onnx", "-int8.onnx")
try:
    quantize_dynamic(ONNX, ONNX_INT8, weight_type=QuantType.QUInt8)
    print("int8 copy:", ONNX_INT8, f"{os.path.getsize(ONNX_INT8) / 1e6:.1f} MB")
except Exception as e:
    print("int8 copy failed (the full-size one is still fine):", e)
    ONNX_INT8 = None

In [ ]:
# 8. Prove how to run the ONNX file outside Python, so the app can copy it exactly.
# Tries the usual recipes and keeps the one whose results match the trained model's own predictions.
import onnxruntime as ort

SESSIONS = {}

def run_onnx(path, img, mean, std, scale):
    if path not in SESSIONS:
        SESSIONS[path] = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    sess = SESSIONS[path]
    inp = sess.get_inputs()[0]
    _, c, h, w = inp.shape
    # A size left open in the file is the model's own resolution
    h = h if isinstance(h, int) else RESOLUTION
    w = w if isinstance(w, int) else RESOLUTION
    x = np.asarray(img.resize((w, h), Image.BILINEAR), dtype=np.float32) / scale
    x = (x - np.array(mean, dtype=np.float32)) / np.array(std, dtype=np.float32)
    x = x.transpose(2, 0, 1)[None]
    outs = sess.run(None, {inp.name: x})
    names = [o.name for o in sess.get_outputs()]
    return dict(zip(names, outs)), inp, names

def decode(outs, names, W, H, threshold):
    boxes = next(v for k, v in outs.items() if "dets" in k or "box" in k)[0]
    logits = next(v for k, v in outs.items() if "label" in k or "logit" in k)[0]
    probs = 1 / (1 + np.exp(-logits))
    cls = probs.argmax(-1); conf = probs.max(-1)
    keep = conf >= threshold
    cx, cy, bw, bh = boxes[keep].T
    xyxy = np.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    return xyxy, cls[keep], conf[keep]

RECIPES = {
    "imagenet": dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], scale=255.0),
    "plain": dict(mean=[0, 0, 0], std=[1, 1, 1], scale=255.0),
}
sample = [os.path.join(MERGED, "test", im["file_name"]) for im in test["images"][:15]] + \
         [os.path.join(EVAL, fr["file"]) for fr in frames[::4]]

def agreement(path, recipe, offset=0):
    """Share of the trained model's finds the ONNX file reproduces: same class, box overlapping by 80%.
    `offset` is the ONNX class index minus the trained model's class id."""
    agree = total = 0
    inp = names = None
    for p in sample:
        img = Image.open(p).convert("RGB"); W, H = img.size
        ref = model.predict(img, threshold=0.5)
        outs, inp, names = run_onnx(path, img, **RECIPES[recipe])
        xyxy, cls, conf = decode(outs, names, W, H, 0.5)
        for rb, rc in zip(ref.xyxy.tolist(), ref.class_id.tolist()):
            total += 1
            if any(int(c) - offset == rc and iou(rb, b.tolist()) >= 0.8 for b, c in zip(xyxy, cls)):
                agree += 1
    return agree / max(1, total), total, inp, names

results = {}
for name in RECIPES:
    for offset in (0, 1, -1):
        score, total, inp, names = agreement(ONNX, name, offset)
        results[f"{name}, class offset {offset}"] = score
        print(f"recipe {name:8s} offset {offset:+d}: {score:6.1%} of the trained model's {total} finds reproduced")
best_key = max(results, key=results.get)
recipe = best_key.split(",")[0]
class_offset = int(best_key.split("offset ")[1])
print("Recipe that matches:", best_key, f"({results[best_key]:.1%})")
if results[best_key] < 0.9:
    print("WARNING: no recipe reproduces 90% of the finds. Tell Claude; the model card has the numbers.")
int8_score = None
if ONNX_INT8:
    int8_score, _, _, _ = agreement(ONNX_INT8, recipe, class_offset)
    print(f"int8 copy reproduces {int8_score:.1%} of the finds")

# The ONNX class index for each name: trained model's class id plus the offset found above
onnx_classes = {}
for cid in range(0, 8):
    n = class_name(cid)
    if n in CLASSES:
        onnx_classes[str(cid + class_offset)] = n

card = {
    "model": f"RF-DETR {MODEL_SIZE}", "licence": "Apache-2.0", "rfdetr": "1.11.2",
    "classes": onnx_classes, "class_offset": class_offset,
    "input": {"name": inp.name, "shape": inp.shape, "layout": "NCHW", "channels": "RGB", **RECIPES[recipe]},
    "outputs": names,
    "decode": "sigmoid on the label logits; best class per query; boxes are cx,cy,w,h as fractions of the input",
    "recipe": recipe, "recipe_agreement": results,
    "int8": os.path.basename(ONNX_INT8) if ONNX_INT8 else None, "int8_agreement": int8_score,
    "test_images": test_report, "deadeye_frames": rows, "dataset": report,
}
with open(os.path.join(ONNX_DIR, "model-card.json"), "w") as f:
    json.dump(card, f, indent=2)
print(json.dumps({k: card[k] for k in ["input", "outputs", "recipe", "recipe_agreement", "int8_agreement"]}, indent=2))

In [ ]:
# 9. Package the result for the laptop
RESULT = os.path.join(DRIVE, "deadeye-rim-detector-result.zip")
with zipfile.ZipFile(RESULT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(os.path.join(ONNX_DIR, "*")):
        if not p.endswith("backbone_model.onnx"):
            z.write(p, os.path.join("onnx", os.path.basename(p)))
    for p in glob.glob(os.path.join(PREVIEWS, "*.jpg")):
        z.write(p, os.path.join("previews", os.path.basename(p)))
print("Done. Download this file from Google Drive (My Drive / deadeye-rim) into Documents/deadeye-bench on the laptop:")
print(" ", RESULT, f"{os.path.getsize(RESULT) / 1e6:.1f} MB")